# Satellite Intelligence - Data Engineering Assignment

Objective:
- Audit data quality issues in parcel readings and parcel metadata.
- Build a reproducible cleaning pipeline.
- Create a clean time-series dataset.
- Analyze NDVI before and after sowing.

In [0]:
#IMPORTS

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    when,
    lower,
    avg,
    round,
    datediff,
    to_date,
    count
)

In [0]:
# Define the actual file paths
METADATA_PATH = "file:/Workspace/Users/srinath.mahto@gmail.com/Carnot assignment/parcel_metadata.csv"
READINGS_PATH = "file:/Workspace/Users/srinath.mahto@gmail.com/Carnot assignment/parcel_readings.csv"

# Read metadata dataframe
metadata_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(METADATA_PATH)
)

# Read readings dataframe
readings_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(READINGS_PATH)
)

# Print the record counts
print(f"Metadata Records : {metadata_df.count()}")
print(f"Reading Records  : {readings_df.count()}")


DATA QUALITY AUDIT


In [0]:
#Date Format Audit

date_audit = readings_df.withColumn(
    "date_format",
    F.when(
        col("date").rlike(r"^\d{4}-\d{2}-\d{2}$"),
        "yyyy-MM-dd"
    )
    .when(
        col("date").rlike(r"^\d{2}/\d{2}/\d{4}$"),
        "dd/MM/yyyy"
    )
    .when(
        col("date").rlike(r"^\d{2}-[A-Za-z]{3}-\d{4}$"),
        "dd-MMM-yyyy"
    )
    .otherwise("unknown")
)

display(
    date_audit.groupBy("date_format").count()
)

In [0]:
#NDVI Validation

invalid_ndvi_count = (
    readings_df
    .filter(
        (col("ndvi_value") < -1) |
        (col("ndvi_value") > 1)
    )
    .count()
)

print(
    f"Invalid NDVI Records: {invalid_ndvi_count}"
)

In [0]:
#Sensor Status Audit

display(
    readings_df
    .groupBy("sensor_status")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
#Missing Values Audit

display(
    readings_df.select([
        count(
            F.when(col(c).isNull(), c)
        ).alias(c)
        for c in readings_df.columns
    ])
)

In [0]:

#Duplicates check

print(
    "Reading Duplicates:",
    readings_df.count()
    - readings_df.dropDuplicates().count()
)

print(
    "Metadata Duplicates:",
    metadata_df.count()
    - metadata_df.dropDuplicates().count()
)

In [0]:
#Referential integrity audit

orphan_parcels = (
    readings_df.join(
        metadata_df,
        "parcel_id",
        "left_anti"
    )
)

print(
    f"Orphan Records: {orphan_parcels.count()}"
)

display(
    orphan_parcels
    .select("parcel_id")
    .distinct()
)


DATA CLEANING


In [0]:
#Standardize Dates

clean_readings = readings_df.withColumn(
    "date",
    F.coalesce(
        F.try_to_date(col("date"), "yyyy-MM-dd"),
        F.try_to_date(col("date"), "dd/MM/yyyy"),
        F.try_to_date(col("date"), "dd-MMM-yyyy")
    )
)

In [0]:
# Check the total number of records after standardizing
print(f"Total Records: {clean_readings.count()}")

# Verify if any dates failed to parse (should be 0)
unparsed_count = clean_readings.filter(col("date").isNull()).count()
print(f"Unparsed/Null Dates: {unparsed_count}")

In [0]:
#Remove Invalid NDVI Values

clean_readings = clean_readings.filter(
    (col("ndvi_value") >= -1)
    &
    (col("ndvi_value") <= 1)
)

In [0]:
# Databricks Python cell

original_count = readings_df.count()

post_ndvi_count = clean_readings.count()
rows_removed = original_count - post_ndvi_count

print(f"Original Records: {original_count}")
print(f"Total Records after NDVI cleaning: {post_ndvi_count}")
print(f"Total NDVI Records Removed: {rows_removed}")

In [0]:
#Standardize Sensor Status

clean_readings = clean_readings.withColumn(
    "sensor_status_clean",
    when(
        lower(col("sensor_status")) == "ok",
        "good"
    )
    .when(
        lower(col("sensor_status")) == "error",
        "bad"
    )
    .otherwise("bad")
)


In [0]:
# 1. Check total records to ensure no rows were accidentally dropped
post_sensor_count = clean_readings.count()
print(f"Total Records after Sensor cleaning: {post_sensor_count}")

# 2. Verify the breakdown of the new 'sensor_status_clean' column
display(
    clean_readings
    .groupBy("sensor_status_clean")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
#Join with Metadata

cleaned_df = (
    clean_readings.join(
        metadata_df,
        "parcel_id",
        "inner"
    )
)




# Inspecting the first few rows to ensure metadata columns are attached correctly
display(cleaned_df)

In [0]:
#Validation

print(
    f"Final Cleaned Records: {cleaned_df.count()}"
)

 Cleaning complete

 Proceeding with Ananlysis

In [0]:
#Keep Trusted Sensor Readings

analysis_df = cleaned_df.filter(
    col("sensor_status_clean") == "good"
)

print(f"Total Trusted Records: {analysis_df.count()}")

# 2. To inspect the actual data rows
display(analysis_df)

In [0]:
#Calculate Relative Days from Sowing

analysis_df = analysis_df.withColumn(
    "days_from_sowing",
    datediff(
        col("date"),
        col("sowing_date")
    )
)

# 1. Verifying the schema to ensure the new column is present as an IntegerType
analysis_df.printSchema()

# 2. Inspecting a few records to confirm the day differences look correct
display(analysis_df.select("parcel_id", "date", "sowing_date", "days_from_sowing"))

In [0]:
#Create Analysis Windows

before_df = analysis_df.filter(
    (col("days_from_sowing") >= -30)
    &
    (col("days_from_sowing") < 0)
)

after_df = analysis_df.filter(
    (col("days_from_sowing") >= 0)
    &
    (col("days_from_sowing") <= 30)
)

In [0]:
#Calculate Mean NDVI

before_crop = (
    before_df
    .groupBy("crop_type")
    .agg(
        round(
            avg("ndvi_value"),
            4
        ).alias("mean_ndvi_before")
    )
)

after_crop = (
    after_df
    .groupBy("crop_type")
    .agg(
        round(
            avg("ndvi_value"),
            4
        ).alias("mean_ndvi_after")
    )
)



# View the Mean NDVI before sowing
print("Mean NDVI (30 Days Before Sowing):")
display(before_crop)

# View the Mean NDVI after sowing
print("Mean NDVI (30 Days After Sowing):")
display(after_crop)

In [0]:
#calculating parcel counts

parcel_counts = (
    analysis_df
    .select(
        "crop_type",
        "parcel_id"
    )
    .distinct()
    .groupBy("crop_type")
    .count()
    .withColumnRenamed(
        "count",
        "n_parcels"
    )
)



#calculating parcel counts
parcel_counts = (
    analysis_df
    .select(
        "crop_type",
        "parcel_id"
    )
    .distinct()
    .groupBy("crop_type")
    .count()
    .withColumnRenamed(
        "count",
        "n_parcels"
    )
)

display(parcel_counts)

In [0]:
#Final output

final_analysis = (
    before_crop
    .join(after_crop, "crop_type")
    .join(parcel_counts, "crop_type")
)

display(final_analysis)

In [0]:
#validating the results

final_analysis.orderBy("crop_type").show(truncate=False)

## Summary

Records Loaded: 3447

Records Removed:
- Invalid NDVI Records: 104
- Orphan Records: 40

Final Clean Dataset:
3303 records

Analysis completed using only trusted sensor readings.

In [0]:
print(f"Clean Dataset Count : {cleaned_df.count()}")
print(f"Trusted Records     : {analysis_df.count()}")
print(f"Bad Sensor Records  : {cleaned_df.filter(col('sensor_status_clean')=='bad').count()}")

In [0]:
pandas_df = cleaned_df.toPandas()

In [0]:
pandas_df.to_csv(
    "cleaned_parcel_timeseries.csv",
    index=False
)

In [0]:
pandas_df.to_csv(
    "/Workspace/Users/srinath.mahto@gmail.com//cleaned_parcel_timeseries.csv",
    index=False
)